In [1]:
# pip install interpret --break-system-packages

# Fase 5 · M05: MLP + EBM — VERSIÓN PRUEBA

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

## ⚡ Versión prueba

Subset estratificado 20% del train · 3-Fold CV · Sin Optuna · Sin serialización definitiva.
Objetivo: verificar que MLP y EBM funcionan end-to-end antes de lanzar `f5_m05_mlp_ebm.ipynb`.

## 📤 Genera

`docs/html/fase5/m05_mlp_ebm_prueba.html`

## ➡️ Siguiente

Si todo es correcto → `f5_m05_mlp_ebm.ipynb` con `FORZAR_OPTUNA = True`


In [2]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from interpret.glassbox import ExplainableBoostingClassifier
    EBM_OK = True
except ImportError:
    EBM_OK = False
    print('⚠️  InterpretML no instalado — pip install interpret --break-system-packages')

warnings.filterwarnings('ignore')

# ROOT detection
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(6):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina

RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_HTML_F5])

RANDOM_STATE  = 42
N_SPLITS_CV   = 3
SUBSET_RATIO  = 0.20
FAMILIA       = 'mlp_ebm'
ESTRATEGIAS   = ['none', 'balanced', 'smote']
MODELOS_M05   = ['MLP', 'EBM']
COLOR         = '#3182ce'
COLOR_MLP     = '#3182ce'
COLOR_EBM     = '#805ad5'
colores_2     = {'MLP': COLOR_MLP, 'EBM': COLOR_EBM}
fmt           = formato_numero_es

info_entorno()
print(f'\n⚡ Modo PRUEBA — subset {int(SUBSET_RATIO*100)}% · {N_SPLITS_CV}-Fold · sin Optuna')
print(f'🧠 InterpretML (EBM): {EBM_OK}')


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [3]:
# ============================================================================
# CELDA 2: CARGA DE DATOS + SUBSET ESTRATIFICADO 20%
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    print('✅ Preprocesados generados')

# Subset estratificado 20%
sss = StratifiedShuffleSplit(n_splits=1, test_size=1-SUBSET_RATIO, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_train_prep = X_train_prep.iloc[idx_sub]
y_train      = y_train.iloc[idx_sub]

FEATURE_NAMES = list(X_train.columns)
N_FEATURES    = len(FEATURE_NAMES)

print(f'Subset prueba : {fmt(len(X_train_prep))} filas ({int(SUBSET_RATIO*100)}%)  abandono={y_train.mean()*100:.1f}%')
print(f'X_test        : {fmt(len(X_test))} × {N_FEATURES}  abandono={y_test.mean()*100:.1f}%')


✅ Preprocesados cargados desde disco
Subset prueba : 5.379 filas (20%)  abandono=29.2%
X_test        : 6.725 × 19  abandono=29.2%


In [4]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo': nombre, 'estrategia': estrategia, 'familia': FAMILIA,
        'f1_mean': cv_res['test_f1'].mean(), 'f1_std': cv_res['test_f1'].std(),
        'auc_mean': cv_res['test_roc_auc'].mean(), 'auc_std': cv_res['test_roc_auc'].std(),
        'precision_mean': cv_res['test_precision'].mean(),
        'recall_mean': cv_res['test_recall'].mean(),
        'accuracy_mean': cv_res['test_accuracy'].mean(),
        'tiempo_s': round(t_total, 3), 'memoria_mb': round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    y_pred  = pipe_fit.predict(X_te)
    y_proba = pipe_fit.predict_proba(X_te)[:, 1]
    return {
        'modelo': nombre, 'estrategia': estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'auc_pr_test'    : round(average_precision_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


print('✅ Funciones definidas')


✅ Funciones definidas


In [5]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS POR DEFECTO (sin Optuna en versión prueba)
# ============================================================================

PARAMS_OPTUNA = {
    'MLP': {
        'hidden_layer_sizes' : (100, 50),
        'activation'         : 'relu',
        'solver'             : 'adam',
        'alpha'              : 0.0001,
        'learning_rate_init' : 0.001,
        'max_iter'           : 300,
        'early_stopping'     : True,
        'validation_fraction': 0.1,
    },
    'EBM': {
        'max_bins'    : 256,
        'interactions': 10,
        'learning_rate': 0.01,
        'max_rounds'  : 3000,
    },
}
AUC_OPTUNA = {'MLP': None, 'EBM': None}
mejores_params = PARAMS_OPTUNA

print('⚡ Params por defecto (versión prueba — sin Optuna)')
for nombre, params in mejores_params.items():
    print(f'   {nombre}: {params}')


⚡ Params por defecto (versión prueba — sin Optuna)
   MLP: {'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0001, 'learning_rate_init': 0.001, 'max_iter': 300, 'early_stopping': True, 'validation_fraction': 0.1}
   EBM: {'max_bins': 256, 'interactions': 10, 'learning_rate': 0.01, 'max_rounds': 3000}


In [6]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 6 COMBINACIONES (2 modelos × 3 estrategias)
# ============================================================================

def construir_modelo(nombre: str, estrategia: str):
    p = mejores_params[nombre]
    if nombre == 'MLP':
        return MLPClassifier(**p, random_state=RANDOM_STATE)
    elif nombre == 'EBM':
        if not EBM_OK:
            raise ImportError('InterpretML no instalado')
        return ExplainableBoostingClassifier(**p, random_state=RANDOM_STATE)


modelos_fit     = {}
resultados_cv   = []
resultados_test = []
emisiones       = None

print('=' * 60)
print('ENTRENAMIENTO PRUEBA — 2 modelos × 3 estrategias = 6 combinaciones')
print('=' * 60)
t0_total = time.time()

for nombre in MODELOS_M05:
    print(f'  {nombre}...', end=' ', flush=True)
    for estrategia in ESTRATEGIAS:
        clave  = f'{nombre}__{estrategia}'
        modelo = construir_modelo(nombre, estrategia)
        res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                            X_train_prep, y_train, estrategia)
        resultados_cv.append(res_cv)
        pipe_final = construir_pipeline(modelo, estrategia)
        pipe_final.fit(X_train_prep, y_train)
        modelos_fit[clave] = pipe_final
        # sin guardar pkl en versión prueba
        res_test = calcular_metricas_test(nombre, pipe_final,
                                          X_test_prep, y_test, estrategia)
        resultados_test.append(res_test)
    print('✅')

df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
df_test = pd.DataFrame(resultados_test)
df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])

MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
mejor_clave      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[mejor_clave]

print(f'\n⏱  Tiempo total: {time.time()-t0_total:.1f}s')
print(f'\n🏆 Mejor (orientativo): {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')


ENTRENAMIENTO PRUEBA — 2 modelos × 3 estrategias = 6 combinaciones
✅ MLP... 
✅ EBM... 

⏱  Tiempo total: 856.6s

🏆 Mejor (orientativo): EBM + none
   AUC CV = 0.9116 ± 0.0052
   F1  CV = 0.7539 ± 0.0161


In [7]:
# ============================================================================
# CELDA 6: TABLA RESULTADOS
# ============================================================================

print(df_cv[['modelo','estrategia','auc_mean','auc_std','f1_mean',
             'precision_mean','recall_mean','tiempo_s']]
      .to_string(index=False, float_format='{:.4f}'.format))


modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
   EBM       none    0.9116   0.0052   0.7539          0.8233       0.6955  106.2040
   EBM   balanced    0.9116   0.0052   0.7539          0.8233       0.6955  111.3790
   EBM      smote    0.9080   0.0048   0.7545          0.7829       0.7285  329.1150
   MLP      smote    0.8867   0.0063   0.7313          0.6914       0.7769    9.4750
   MLP       none    0.8816   0.0018   0.7090          0.7664       0.6599    3.5970
   MLP   balanced    0.8816   0.0018   0.7090          0.7664       0.6599    5.7570


In [8]:
# ============================================================================
# CELDA 7: ARQUITECTURA MLP (capas vs AUC) — versión reducida
# ============================================================================

graficos_b64 = {}

arquitecturas = {
    '(50,)'     : (50,),
    '(100,)'    : (100,),
    '(100, 50)' : (100, 50),
    '(100,50,25)': (100, 50, 25),
}

cv_arq = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
resultados_arq = {}
print('Evaluando arquitecturas...', end=' ', flush=True)
t0 = time.time()
for nombre_arq, hidden in arquitecturas.items():
    mlp_arq = MLPClassifier(
        hidden_layer_sizes=hidden, activation='relu',
        alpha=0.0001, max_iter=200, early_stopping=True, random_state=RANDOM_STATE
    )
    pipe_arq = ImbPipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)), ('model', mlp_arq)
    ])
    cv_r = cross_validate(pipe_arq, X_train_prep, y_train,
                          cv=cv_arq, scoring='roc_auc', n_jobs=-1)
    resultados_arq[nombre_arq] = cv_r['test_score'].mean()

print(f'✅  ({time.time()-t0:.0f}s)')

nombres_arq = list(resultados_arq.keys())
aucs_arq    = list(resultados_arq.values())
best_arq    = nombres_arq[int(np.argmax(aucs_arq))]

fig, ax = plt.subplots(figsize=(9, 4))
colores_arq = [COLOR_MLP if n != best_arq else '#e53e3e' for n in nombres_arq]
bars = ax.bar(nombres_arq, aucs_arq, color=colores_arq, edgecolor='white')
ax.set_ylim(max(0.5, min(aucs_arq)-0.02), min(1.0, max(aucs_arq)+0.03))
ax.set_ylabel('AUC-ROC (3-Fold CV)')
ax.set_title('MLP — AUC por arquitectura (prueba)', fontweight='bold')
for bar, val in zip(bars, aucs_arq):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['arquitectura_mlp'] = figura_a_base64(fig)
plt.close(fig)
print(f'Mejor arquitectura: {best_arq}  AUC={max(aucs_arq):.4f}')


Evaluando arquitecturas... ✅  (17s)
Mejor arquitectura: (50,)  AUC=0.8889


In [9]:
# ============================================================================
# CELDA 8: FEATURE IMPORTANCE EBM
# ============================================================================

try:
    mejor_est_ebm = (df_cv[df_cv['modelo']=='EBM']
                     .sort_values('auc_mean', ascending=False).iloc[0]['estrategia'])
    pipe_ebm  = modelos_fit[f'EBM__{mejor_est_ebm}']
    ebm_model = pipe_ebm.named_steps.get('model') or pipe_ebm[-1]

    importancias_ebm  = ebm_model.term_importances()
    feature_names_ebm = ebm_model.term_names_
    idx_top    = np.argsort(importancias_ebm)[-min(10, len(importancias_ebm)):]
    nombres_top = [feature_names_ebm[i] for i in idx_top]
    valores_top = importancias_ebm[idx_top]

    colores_ebm = ['#e53e3e' if v == valores_top.max() else COLOR_EBM for v in valores_top]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(nombres_top, valores_top, color=colores_ebm, height=0.6)
    ax.set_xlabel('Importancia global EBM')
    ax.set_title('EBM — Feature importance global (prueba)', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    graficos_b64['importancia_ebm'] = figura_a_base64(fig)
    plt.close(fig)
    print(f'✅ Feature importance EBM — top: {nombres_top[-1]}  ({valores_top[-1]:.4f})')
except Exception as e:
    print(f'⚠️  Feature importance EBM no disponible: {e}')
    graficos_b64['importancia_ebm'] = None


✅ Feature importance EBM — top: n_anios_beca  (1.0094)


In [10]:
# ============================================================================
# CELDA 9: CALIBRACIÓN + ROC + PR CURVE
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ax_cal, ax_roc, ax_pr = axes

ax_cal.plot([0,1],[0,1],'k--',linewidth=1.2,label='Perfecta')
ax_roc.plot([0,1],[0,1],'k--',linewidth=1)
baseline_pr = y_test.mean()
ax_pr.axhline(baseline_pr, color='gray', linestyle='--', linewidth=1,
              label=f'Baseline ({baseline_pr:.2f})')

for nombre_m in MODELOS_M05:
    mejor_est = (df_cv[df_cv['modelo']==nombre_m]
                 .sort_values('auc_mean',ascending=False).iloc[0]['estrategia'])
    pipe_m  = modelos_fit[f'{nombre_m}__{mejor_est}']
    y_proba = pipe_m.predict_proba(X_test_prep)[:,1]
    color_m = colores_2.get(nombre_m, COLOR)

    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-', color=color_m,
                label=f'{nombre_m}', linewidth=1.8, markersize=5)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=color_m, label=f'{nombre_m} AUC={auc_v:.3f}', linewidth=2)

    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    auc_pr = average_precision_score(y_test, y_proba)
    ax_pr.plot(rec, prec, color=color_m, label=f'{nombre_m} AUC-PR={auc_pr:.3f}', linewidth=2)

for ax, titulo in zip([ax_cal, ax_roc, ax_pr],
                      ['Calibración','Curvas ROC','Curva PR (desbalance 70/30)']):
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax_cal.set_xlabel('Prob. predicha');  ax_cal.set_ylabel('Fracción positivos reales')
ax_roc.set_xlabel('FPR');             ax_roc.set_ylabel('TPR')
ax_pr.set_xlabel('Recall');           ax_pr.set_ylabel('Precision')

plt.tight_layout()
graficos_b64['calibracion_roc_pr'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Calibración + ROC + PR generados')


✅ Calibración + ROC + PR generados


In [11]:
# ============================================================================
# CELDA 10: MATRIZ DE CONFUSIÓN + COMPARATIVA AUC
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[0]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0,1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0,1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Confusión — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}', fontweight='bold')
ax_cm.set_ylabel('Real'); ax_cm.set_xlabel('Predicho')

ax_bar = axes[1]
aucs_cv     = [df_cv[df_cv['modelo']==m]['auc_mean'].max() for m in MODELOS_M05]
colores_bar = [colores_2.get(m, COLOR) for m in MODELOS_M05]
bars = ax_bar.bar(MODELOS_M05, aucs_cv, color=colores_bar, edgecolor='white', width=0.4)
ax_bar.set_ylim(max(0.5, min(aucs_cv)-0.02), min(1.0, max(aucs_cv)+0.02))
ax_bar.set_ylabel('AUC-ROC (CV)')
ax_bar.set_title('MLP vs EBM — AUC', fontweight='bold')
for bar, val in zip(bars, aucs_cv):
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['cm_comparativa'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Matriz de confusión + comparativa generados')
print()
print(classification_report(y_test, y_pred_mejor, target_names=['Continúa','Abandona']))


✅ Matriz de confusión + comparativa generados

              precision    recall  f1-score   support

    Continúa       0.89      0.93      0.91      4758
    Abandona       0.82      0.72      0.77      1967

    accuracy                           0.87      6725
   macro avg       0.85      0.83      0.84      6725
weighted avg       0.87      0.87      0.87      6725



In [12]:
# ============================================================================
# CELDA 11: GENERAR HTML
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm05_mlp_ebm_prueba.html'

aviso_prueba = '''
<div style="background:#fff3cd;border:1px solid #ffc107;border-radius:8px;
            padding:14px 18px;margin-bottom:20px;">
  <strong>⚡ Versión prueba</strong> — Subset 20% del train · 3-Fold CV · Sin Optuna<br>
  Resultados orientativos. Ejecutar <code>f5_m05_mlp_ebm.ipynb</code> para los artefactos definitivos.
</div>'''

# KPIs
kpis = [
    {'valor': '2',                                  'titulo': 'Modelos'},
    {'valor': '3',                                  'titulo': 'Estrategias'},
    {'valor': '6',                                  'titulo': 'Combinaciones'},
    {'valor': '❌ Prueba',                           'titulo': 'Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}',   'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',    'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

# Tabla CV
filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}"><td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} ± {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td><td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td><td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean±std)</th>'
    '<th>F1</th><th>Precisión</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

def img_tag(key):
    b64 = graficos_b64.get(key)
    if not b64:
        return '<p><em>Gráfico no disponible</em></p>'
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">'

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + aviso_prueba + kpis_html + '</section>'
    '<section class="seccion"><h2>Resultados 3-Fold CV — 6 combinaciones (orientativo)</h2>' + tabla_cv + '</section>'
    '<section class="seccion"><h2>Arquitectura MLP — capas vs AUC</h2>'
    + img_tag('arquitectura_mlp') + '</section>'
    '<section class="seccion"><h2>EBM — Feature importance global</h2>'
    + img_tag('importancia_ebm') + '</section>'
    '<section class="seccion"><h2>Calibración + ROC + Curva PR</h2>'
    + img_tag('calibracion_roc_pr') + '</section>'
    '<section class="seccion"><h2>Matriz de confusión + comparativa AUC</h2>'
    + img_tag('cm_comparativa') + '</section>'
)

render_pagina(
    'f5_m05_mlp_ebm_prueba.ipynb',
    secciones,
    RUTA_HTML_SALIDA,
    carpeta_notebook='fase5_modelado'
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')



✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m05_mlp_ebm_prueba.html


In [13]:
# ============================================================================
# CELDA 12: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m05_mlp_ebm_prueba')
print('=' * 60)
print(f'Subset      : {fmt(len(X_train_prep))} filas ({int(SUBSET_RATIO*100)}% train)')
print(f'Modelos     : MLPClassifier · ExplainableBoostingMachine')
print(f'Estrategias : none · balanced · smote  (6 combinaciones)')
print(f'CV folds    : {N_SPLITS_CV}  (reducido)')
print( 'Optuna      : ❌ No ejecutado (params por defecto)')
print()
print(f'🏆 Mejor (orientativo): {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('✅ Pipeline verificado — listo para lanzar f5_m05_mlp_ebm.ipynb')


RESUMEN FINAL — f5_m05_mlp_ebm_prueba
Subset      : 5.379 filas (20% train)
Modelos     : MLPClassifier · ExplainableBoostingMachine
Estrategias : none · balanced · smote  (6 combinaciones)
CV folds    : 3  (reducido)
Optuna      : ❌ No ejecutado (params por defecto)

🏆 Mejor (orientativo): EBM + none
   AUC CV = 0.9116 ± 0.0052
   F1  CV = 0.7539 ± 0.0161

✅ Pipeline verificado — listo para lanzar f5_m05_mlp_ebm.ipynb
